# CMP4509 — MNIST üzerinde PGD ile dayanıklı öğrenme

Broski, ilk hedefimiz bütün zinciri küçük bir deneyle çalıştırmak: **veri → CNN → normal eğitim → saldırı → PGD eğitimi → daha güçlü test → CSV/grafik**.

Bu notebook **kendi kaynak kodunu içerir**. Colab/Kaggle'da başka bir depo klonlamadan, yukarıdan aşağı çalışır. Varsayılan `smoke`, küçük bir çalışma kontrolüdür; bilimsel sonuç değildir. Asıl deneyler GPU'da çalıştırılınca gerçek sayıları raporlayacağız.

Kaynaklar: [Madry ve ark., makale](https://arxiv.org/abs/1706.06083), [yazarların MNIST uygulaması](https://github.com/MadryLab/mnist_challenge).

**Colab:** Runtime → Change runtime type → GPU. **Kaggle:** Settings → Accelerator → GPU, ilk MNIST indirmesi için Internet açık. GPU kotası ve erişim garanti değildir. `smoke` CPU'da çalışabilir; `pilot/main` için bu notebook CUDA GPU ister.

Rapor: **13 Aralık 2026, 23:59**; gerçek `CMP4509_report_template.zip` şablonu kullanılacak. Sunum: **30 Kasım veya 7 Aralık**, QA dahil **10 dakika**. Public GitHub bağlantısı raporda bulunmalı.


## 1 · Ortamı kontrol et

Önceden kurulmuş GPU destekli PyTorch'u koruyoruz. Yalnız eksik paketler kurulur. Birbiriyle uyumsuz torch/torchvision varsa hücre açık hata verir; temiz bir GPU oturumunda yeniden başlatmak genellikle en kolay başlangıçtır.


In [ ]:
import importlib.util
import subprocess
import sys

missing_torch = importlib.util.find_spec("torch") is None
missing_vision = importlib.util.find_spec("torchvision") is None
if missing_torch:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch>=2.5,<3", "torchvision>=0.20,<1"])
elif missing_vision:
    raise RuntimeError(
        "PyTorch var ama torchvision eksik. Torch sürümünüzle uyumlu torchvision kurun "
        "veya temiz Colab/Kaggle GPU oturumu açın; GPU paketini otomatik değiştirmiyoruz."
    )
for package, requirement in {"numpy": "numpy>=1.26,<3", "matplotlib": "matplotlib>=3.8,<4", "pandas": "pandas>=2.2,<4"}.items():
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", requirement])

try:
    import torch
    import torchvision
    import numpy as np
    import matplotlib
    import pandas as pd
except Exception as exc:
    raise RuntimeError(
        "Paketler uyumsuz veya eksik. Temiz bir Colab/Kaggle GPU oturumunda yeniden deneyin. "
        "Yerelde PyTorch'un resmî kurulumundan birbiriyle uyumlu torch/torchvision seçin."
    ) from exc

print("Python:", sys.version.split()[0])
print("torch:", torch.__version__, "torchvision:", torchvision.__version__)
print("CUDA GPU:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2 · Deney ayarları ve kalıcı kayıt

İlk çalıştırmada hiçbir şeyi değiştirmeden ilerle. Sonra `pilot` ile öğrenme ve süreyi gözle. `main`, her iki eğitim yöntemi için üç seed kullanır. **Eğitim ve model seçimi doğrulama setiyle yapılır.** `EPOCHS_PER_CALL=1` ile ücretsiz GPU oturumlarında eğitimi birer epoch parçalara ayırabilirsin; test bütün epoch'lar tamamlanana kadar bekler. `pilot/main` testini ancak bütün ayarları dondurunca `PROTOCOL_FROZEN = True` ile aç.

Final rapor için `CONFIG_NAME = "main"`, `PROTOCOL_FROZEN = True`, `FINAL_STRONG_EVAL = True` kullan: 100 PGD adımı × 20 restart. Önceden eğitilmiş checkpoint varsa eğitim son tamamlanan epoch'tan devam eder; tamamlanmışsa yalnız değerlendirme ilerler.

Colab'da uzun koşudan önce `USE_GOOGLE_DRIVE = True` yap. Drive bağlamak kullanıcı girişini gerektirebilir. Kaggle çıktıları `/kaggle/working` altında saklanır; oturum sonunda **Save Version** yap. Aynı kayıt kökünde config'ini değiştirme: değiştirilmiş deney için yeni config adı/kayıt kökü kullan.


In [ ]:
from pathlib import Path
import os

CONFIG_NAME = "smoke"                    # "smoke", "pilot", "main"
MODES = ["baseline", "adversarial"]
SEEDS = [0, 1, 2] if CONFIG_NAME == "main" else [0]
PROTOCOL_FROZEN = False                   # Pilot/main testi: kararlar bittikten sonra True
FINAL_STRONG_EVAL = False                 # Final: PGD100 x 20 restart
USE_GOOGLE_DRIVE = False                  # Colab uzun koşularında True önerilir
RESUME = True                            # last.pt varsa oradan devam et
EPOCHS_PER_CALL = 0                       # 0=hepsi; örn. 1 ile oturum başına bir epoch
SYNTHETIC = False                         # Yalnız çevrimdışı smoke; MNIST sonucu değildir

assert CONFIG_NAME in {"smoke", "pilot", "main"}
assert MODES and set(MODES) <= {"baseline", "adversarial"}
assert SEEDS and len(SEEDS) == len(set(SEEDS))
if CONFIG_NAME != "smoke" and not torch.cuda.is_available():
    raise RuntimeError("pilot/main için CUDA GPU açın veya CONFIG_NAME='smoke' ile başlayın.")
if SYNTHETIC and CONFIG_NAME != "smoke":
    raise ValueError("Sentetik veri yalnız smoke kontrolü içindir.")
if FINAL_STRONG_EVAL and not PROTOCOL_FROZEN:
    raise ValueError("Final test için önce ayarları dondurun ve PROTOCOL_FROZEN=True yapın.")

try:
    IS_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IS_COLAB = False
IS_KAGGLE = Path("/kaggle/working").exists()
if "_NOTEBOOK_BASE_DIR" not in globals():
    _NOTEBOOK_BASE_DIR = Path.cwd()
BASE_DIR = Path("/content") if IS_COLAB else (Path("/kaggle/working") if IS_KAGGLE else _NOTEBOOK_BASE_DIR)
PROJECT_DIR = BASE_DIR / "madry_mnist_project"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
if USE_GOOGLE_DRIVE:
    if not IS_COLAB:
        raise RuntimeError("USE_GOOGLE_DRIVE yalnız Colab içindir.")
    from google.colab import drive
    drive.mount("/content/drive")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/CMP4509_madry")
else:
    ARTIFACT_ROOT = BASE_DIR / "madry_mnist_outputs"
RUN_ROOT = ARTIFACT_ROOT / ("synthetic_runs" if SYNTHETIC else "runs")
RESULTS_DIR = ARTIFACT_ROOT / ("synthetic_results" if SYNTHETIC else "results")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)
for name in ["mnist_robust", "configs", "docs"]:
    Path(name).mkdir(exist_ok=True)

RUN_CONFIG_NAME = CONFIG_NAME + ("_synthetic" if SYNTHETIC else "")
RUN_TEST_EVALUATION = CONFIG_NAME == "smoke" or PROTOCOL_FROZEN
print("Kaynak:", PROJECT_DIR)
print("Kalıcı/indirilecek çıktılar:", ARTIFACT_ROOT)
print("Config:", CONFIG_NAME, "| Modeller:", MODES, "| Seed'ler:", SEEDS)
print("Test açık mı?", RUN_TEST_EVALUATION, "| Final güçlü test?", FINAL_STRONG_EVAL)
print("Ekran görüntüsü zamanı: bu ayarlar ve GPU bilgisi deney kaydınız olsun.")


## 3 · Kaynak kodunu oluştur

Aşağıdaki hücreler projedeki kaynakları görünür biçimde yazar. `Run all` için değiştirmeye gerek yok. Öğrenirken özellikle model, saldırı ve eğitim dosyalarını açıp birlikte okuyabiliriz.

Makalenin hedefi:

\[
\min_\theta \; \mathbb{E}_{(x,y)} [ \max_{\|\delta\|_\infty\leq 0.3} \mathcal{L}(f_\theta(x+\delta),y)] .
\]

İçte PGD, mevcut modeli zorlayacak bir görüntü arar. Dışta optimizer, modelin bu zor görüntüdeki kaybını düşürür. Piksel ölçeği `[0,1]` olduğu için ε=0.3 bu ölçekte uygulanır. Her PGD adımından sonra hem ε sınırına hem `[0,1]` aralığına geri dönülür.

Resmî kod: 32/64 convolution, 1024 gizli birim, Adam `1e-4`, batch 50, 100.000 güncelleme; eğitim PGD40, α=.01, ε=.3. Bizim `main` başlangıcımız daha küçük ağ/10 epoch/Adam `1e-3`/batch128 kullanır. **Bu farklar raporlanır; birebir sayısal tekrar iddiası kurulmaz.**


In [ ]:
%%writefile mnist_robust/__init__.py
"""MNIST üzerinde küçük ve tekrar çalıştırılabilir adversarial training projesi."""

from .attacks import fgsm, pgd_linf
from .model import SmallCNN

__all__ = ["SmallCNN", "fgsm", "pgd_linf"]
__version__ = "0.1.0"


In [ ]:
%%writefile mnist_robust/attacks.py
"""Untargeted FGSM ve L-infinity PGD; tehdit modeli ham [0, 1] piksellerde.

Değerlendirmede her örnek için bulunan herhangi bir yanlış sınıflandırmayı
koruruz. Böylece son iterasyonun tesadüfen tekrar doğru sınıflandırılması,
ölçülen robust accuracy değerini yapay olarak yükseltmez.
"""

from contextlib import contextmanager
import math
from numbers import Real

import torch
from torch import nn
from torch.nn import functional as F


@contextmanager
def _attack_mode(model: nn.Module):
    # Mixed train/eval alt modül ayarlarını da geri yükle. BN istatistikleri
    # saldırı üretilirken güncellenmez; dış eğitim adımı bağımsız kalır.
    modes = [(module, module.training) for module in model.modules()]
    model.eval()
    try:
        yield
    finally:
        for module, training in modes:
            module.training = training


def _nonnegative_finite(name, value):
    if isinstance(value, bool) or not isinstance(value, Real):
        raise ValueError(f"{name} sonlu ve negatif olmayan bir sayı olmalı.")
    if not math.isfinite(value) or value < 0:
        raise ValueError(f"{name} sonlu ve negatif olmayan bir sayı olmalı.")


def _validate_inputs(x, y, eps):
    _nonnegative_finite("eps", eps)
    if not isinstance(x, torch.Tensor) or not x.is_floating_point():
        raise ValueError("x kayan noktalı bir Tensor olmalı.")
    if x.ndim < 2 or x.shape[0] == 0:
        raise ValueError("x boş olmayan bir batch içermeli.")
    if not isinstance(y, torch.Tensor) or y.ndim != 1 or len(y) != len(x):
        raise ValueError("y her giriş için bir sınıf etiketi içermeli.")
    if y.dtype != torch.long or y.device != x.device:
        raise ValueError("y torch.long olmalı ve x ile aynı cihazda bulunmalı.")
    if not torch.isfinite(x).all() or torch.any(x < 0) or torch.any(x > 1):
        raise ValueError("Saldırı ham, sonlu [0, 1] pikseller bekliyor.")


def _project(candidate, clean, eps):
    return torch.maximum(torch.minimum(candidate, clean + eps), clean - eps).clamp(0, 1)


def _select_candidate(best, best_loss, best_wrong, candidate, logits, losses, y):
    """Önce başarılı saldırı, sonra cross-entropy: örnek bazında seçim."""
    with torch.no_grad():
        wrong = logits.argmax(dim=1).ne(y)
        replace = (wrong & ~best_wrong) | ((wrong == best_wrong) & (losses > best_loss))
        broadcast = replace.reshape((-1,) + (1,) * (candidate.ndim - 1))
        best = torch.where(broadcast, candidate.detach(), best)
        best_loss = torch.where(replace, losses.detach(), best_loss)
        best_wrong = torch.where(replace, wrong, best_wrong)
    return best, best_loss, best_wrong


def fgsm(model: nn.Module, x: torch.Tensor, y: torch.Tensor, eps: float):
    """Bir gradient-sign adımı; modelin modu ve parametre .grad'ları korunur."""
    _validate_inputs(x, y, eps)
    clean = x.detach()
    if eps == 0:
        return clean.clone()
    with _attack_mode(model), torch.enable_grad():
        candidate = clean.clone().requires_grad_(True)
        loss = F.cross_entropy(model(candidate), y, reduction="sum")
        gradient, = torch.autograd.grad(loss, candidate, only_inputs=True)
        return _project(candidate.detach() + eps * gradient.sign(), clean, eps).detach()


def pgd_linf(
    model: nn.Module,
    x: torch.Tensor,
    y: torch.Tensor,
    eps: float = 0.3,
    alpha: float = 0.01,
    steps: int = 40,
    restarts: int = 1,
    random_start: bool = True,
    track_best: bool = True,
):
    """Untargeted L-infinity PGD.

    track_best=True: temiz girişi, her iterasyonu ve restart'ı karşılaştırır;
    bulunan yanlış sınıflandırmayı her örnek için korur. Değerlendirme içindir.
    track_best=False: tek restart'ın son iterasyonunu döndürür. Adversarial
    training için kullanılır ve restarts=1 gerektirir.

    Dönen Tensor graph'tan ayrıdır. Giriş ya da model parametrelerinin .grad
    alanlarına yazılmaz. torch.no_grad() içinden de çağrılabilir; inference_mode
    içinde çağırmayın, çünkü saldırı giriş türevi gerektirir.
    """
    _validate_inputs(x, y, eps)
    _nonnegative_finite("alpha", alpha)
    for name, value, minimum in (("steps", steps, 0), ("restarts", restarts, 1)):
        if isinstance(value, bool) or not isinstance(value, int) or value < minimum:
            raise ValueError(f"{name} en az {minimum} olan bir tamsayı olmalı.")
    if not isinstance(random_start, bool) or not isinstance(track_best, bool):
        raise ValueError("random_start ve track_best bool olmalı.")
    if not track_best and restarts != 1:
        raise ValueError("track_best=False son iterasyon içindir; restarts=1 kullanın.")
    clean = x.detach()
    if eps == 0:
        return clean.clone()

    with _attack_mode(model):
        if track_best:
            with torch.no_grad():
                logits = model(clean)
                best = clean.clone()
                best_loss = F.cross_entropy(logits, y, reduction="none")
                best_wrong = logits.argmax(dim=1).ne(y)

        for _ in range(restarts):
            if random_start:
                candidate = _project(clean + torch.empty_like(clean).uniform_(-eps, eps), clean, eps)
            else:
                candidate = clean.clone()

            for _ in range(steps):
                with torch.enable_grad():
                    candidate = candidate.detach().requires_grad_(True)
                    logits = model(candidate)
                    losses = F.cross_entropy(logits, y, reduction="none")
                    if track_best:
                        best, best_loss, best_wrong = _select_candidate(
                            best, best_loss, best_wrong, candidate, logits, losses, y
                        )
                    gradient, = torch.autograd.grad(losses.sum(), candidate, only_inputs=True)
                candidate = _project(candidate.detach() + alpha * gradient.sign(), clean, eps).detach()

            if track_best:
                with torch.no_grad():
                    logits = model(candidate)
                    losses = F.cross_entropy(logits, y, reduction="none")
                    best, best_loss, best_wrong = _select_candidate(
                        best, best_loss, best_wrong, candidate, logits, losses, y
                    )

        return (best if track_best else candidate).detach()


In [ ]:
%%writefile mnist_robust/data.py
"""Sabit MNIST train/validation ayrımı; test seti model seçimi için kullanılmaz."""

from pathlib import Path
import random

import numpy as np
import torch
from torch.utils.data import DataLoader, Subset, TensorDataset, random_split


def _seed_worker(_worker_id):
    seed = torch.initial_seed() % (2**32)
    np.random.seed(seed)
    random.seed(seed)


def _positive_int(name, value, allow_none=False):
    if allow_none and value is None:
        return
    if isinstance(value, bool) or not isinstance(value, int) or value <= 0:
        raise ValueError(f"{name} pozitif bir tamsayı olmalı.")


def _limit(dataset, count, name):
    if count is None:
        return dataset
    if count > len(dataset):
        raise ValueError(f"{name}={count}, mevcut örnek sayısını ({len(dataset)}) aşıyor.")
    return Subset(dataset, range(count))


def _synthetic_split(size, seed):
    # Sadece çevrimdışı yazılım smoke testi. MNIST verisi ya da bilimsel sonuç değil.
    # Ayrı RNG'ler, limit küçültüldüğünde örnek/etiket eşleşmesini sabit tutar.
    images = torch.rand(size, 1, 28, 28, generator=torch.Generator().manual_seed(seed))
    labels = torch.randint(10, (size,), generator=torch.Generator().manual_seed(seed + 1000))
    return TensorDataset(images, labels)


def make_loaders(
    data_dir,
    batch_size=128,
    seed=0,
    split_seed=2026,
    val_size=5000,
    train_limit=None,
    val_limit=None,
    test_limit=None,
    num_workers=0,
    synthetic=False,
):
    """{'train', 'val', 'test'} DataLoader sözlüğü döndürür.

    split_seed veriyi ayırır; seed yalnızca batch sırasını/worker RNG'lerini
    yönetir. Dolayısıyla farklı eğitim seed'leri aynı validation verisini görür.
    Piksel ölçeği [0, 1]'dir: epsilon=0.3 doğrudan bu ölçekte uygulanır.
    synthetic=True rastgele görüntü/etiket üretir ve yalnızca smoke test içindir.
    """
    _positive_int("batch_size", batch_size)
    _positive_int("val_size", val_size)
    for name, limit in (("train_limit", train_limit), ("val_limit", val_limit), ("test_limit", test_limit)):
        _positive_int(name, limit, allow_none=True)
    if isinstance(num_workers, bool) or not isinstance(num_workers, int) or num_workers < 0:
        raise ValueError("num_workers negatif olmayan bir tamsayı olmalı.")
    for name, value in (("seed", seed), ("split_seed", split_seed)):
        if isinstance(value, bool) or not isinstance(value, int) or not 0 <= value < 2**32:
            raise ValueError(f"{name}, [0, 2**32) aralığında bir tamsayı olmalı.")

    if synthetic:
        datasets = {
            "train": _synthetic_split(train_limit or 256, split_seed),
            "val": _synthetic_split(val_limit or 64, split_seed + 1),
            "test": _synthetic_split(test_limit or 64, split_seed + 2),
        }
    else:
        # Ortamda torchvision bozuk olsa bile offline attack testleri çalışabilir.
        from torchvision.datasets import MNIST
        from torchvision.transforms import ToTensor

        data_dir = Path(data_dir).expanduser()
        full_train = MNIST(str(data_dir), train=True, download=True, transform=ToTensor())
        if val_size >= len(full_train):
            raise ValueError("val_size, toplam eğitim örnek sayısından küçük olmalı.")
        train, val = random_split(
            full_train,
            [len(full_train) - val_size, val_size],
            generator=torch.Generator().manual_seed(split_seed),
        )
        test = MNIST(str(data_dir), train=False, download=True, transform=ToTensor())
        datasets = {
            "train": _limit(train, train_limit, "train_limit"),
            "val": _limit(val, val_limit, "val_limit"),
            "test": _limit(test, test_limit, "test_limit"),
        }

    return {
        name: DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=name == "train",
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
            worker_init_fn=_seed_worker,
            generator=torch.Generator().manual_seed(seed + index),
            persistent_workers=False,
        )
        for index, (name, dataset) in enumerate(datasets.items())
    }


In [ ]:
%%writefile mnist_robust/experiment.py
"""Train on train/validation only; evaluate frozen checkpoints on the test split."""
import argparse
import csv
import hashlib
import json
import os
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

from .attacks import fgsm, pgd_linf
from .data import make_loaders
from .model import SmallCNN
from .utils import environment_info, resolve_device, seed_everything


def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False) + '\n')


def write_csv(path, rows):
    if not rows:
        return
    with open(path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def rng_state(loader=None):
    return {'python': random.getstate(), 'numpy': np.random.get_state(),
            'torch': torch.get_rng_state(),
            'cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
            'mps': torch.mps.get_rng_state() if torch.backends.mps.is_available() else None,
            'loader': loader.generator.get_state() if loader is not None else None}


def restore_rng(state, loader=None):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch'].cpu())
    if state['cuda'] and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([s.cpu() for s in state['cuda']])
    if state.get('mps') is not None and torch.backends.mps.is_available():
        torch.mps.set_rng_state(state['mps'].cpu())
    if loader is not None and state['loader'] is not None:
        loader.generator.set_state(state['loader'].cpu())


def save_checkpoint(path, state):
    temporary = path.with_suffix('.tmp')
    torch.save(state, temporary)
    os.replace(temporary, path)


def validate(model, loader, device, cfg, seed):
    """Fixed validation attack RNG; validation does not consume training RNG."""
    previous = rng_state()
    seed_everything(seed)
    was_training = model.training
    model.eval()
    total = clean_correct = robust_correct = 0
    try:
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            with torch.no_grad():
                clean = model(x).argmax(1).eq(y)
            adv = pgd_linf(model, x, y, eps=cfg['epsilon'], alpha=cfg['eval_alpha'],
                           steps=cfg['val_steps'], restarts=cfg['val_restarts'])
            with torch.no_grad():
                robust = clean & model(adv).argmax(1).eq(y)
            total += len(y)
            clean_correct += clean.sum().item()
            robust_correct += robust.sum().item()
    finally:
        model.train(was_training)
        restore_rng(previous)
    return clean_correct / total, robust_correct / total


def train(model, loaders, device, cfg, args, run_dir):
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg['lr'])
    start, best_score, history = 0, -1.0, []
    last = run_dir / 'last.pt'
    if last.exists():
        if not args.resume:
            raise FileExistsError(f'{last} exists. Use --resume or a new --output-dir.')
        state = torch.load(last, map_location='cpu', weights_only=False)
        if state['config'] != cfg or state['mode'] != args.mode or state['seed'] != args.seed or state['synthetic'] != args.synthetic:
            raise ValueError('Resume config/mode/seed/dataset differs from saved checkpoint.')
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        for values in optimizer.state.values():
            for key, value in values.items():
                if torch.is_tensor(value):
                    values[key] = value.to(device)
        start, best_score, history = state['epoch'], state['best_score'], state['history']
        restore_rng(state['rng'], loaders['train'])
    end = cfg['epochs']
    if args.epochs_per_call:
        end = min(end, start + args.epochs_per_call)
    for epoch in range(start, end):
        begin = time.monotonic()
        model.train()
        loss_sum, correct, total = 0., 0, 0
        for x, y in loaders['train']:
            x, y = x.to(device), y.to(device)
            if args.mode == 'adversarial':
                x = pgd_linf(model, x, y, eps=cfg['epsilon'], alpha=cfg['train_alpha'],
                             steps=cfg['train_steps'], restarts=1, track_best=False)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(y)
            correct += logits.argmax(1).eq(y).sum().item()
            total += len(y)
        val_clean, val_pgd = validate(model, loaders['val'], device, cfg, cfg['eval_seed'])
        score = val_clean if args.mode == 'baseline' else val_pgd
        improved = score > best_score
        best_score = max(best_score, score)
        row = {'epoch': epoch + 1, 'train_loss': loss_sum / total,
               'train_accuracy': correct / total, 'val_clean_accuracy': val_clean,
               'val_pgd_accuracy': val_pgd, 'seconds': time.monotonic() - begin}
        history.append(row)
        state = {'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                 'epoch': epoch + 1, 'best_score': best_score, 'history': history,
                 'config': cfg, 'seed': args.seed, 'mode': args.mode,
                 'synthetic': args.synthetic, 'rng': rng_state(loaders['train'])}
        save_checkpoint(last, state)
        if improved:
            save_checkpoint(run_dir / 'best.pt', state)
        write_csv(run_dir / 'history.csv', history)
        print(json.dumps(row), flush=True)
    return end >= cfg['epochs']


def plot_examples(path, examples, epsilon):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    x, xf, xp, y, pc, pf, pp = examples
    n = min(len(x), 8)
    fig, axes = plt.subplots(4, n, figsize=(1.7 * n, 7), squeeze=False)
    for i in range(n):
        for row, (images, predictions, label) in enumerate([(x, pc, 'Clean'), (xf, pf, 'FGSM'), (xp, pp, 'PGD')]):
            axes[row, i].imshow(images[i, 0], cmap='gray', vmin=0, vmax=1)
            axes[row, i].set_title(f'{label}: {predictions[i].item()} / y={y[i].item()}', fontsize=8)
        axes[3, i].imshow((xp[i, 0] - x[i, 0]).abs(), cmap='magma', vmin=0, vmax=epsilon)
        axes[3, i].set_title('|PGD − clean|', fontsize=8)
        for row in range(4):
            axes[row, i].axis('off')
    fig.suptitle('First test examples (not cherry-picked); perturbation color range [0, ε]')
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


def evaluate(model, loader, device, cfg, args, run_dir, checkpoint):
    model.eval()
    seed_everything(cfg['eval_seed'])
    steps = args.eval_steps if args.eval_steps is not None else cfg['eval_steps']
    restarts = args.eval_restarts if args.eval_restarts is not None else cfg['eval_restarts']
    if steps <= cfg['train_steps'] or restarts <= 1:
        raise ValueError('Final PGD evaluation requires more steps than training and >1 restart.')
    protocol = {'epsilon': cfg['epsilon'], 'alpha': cfg['eval_alpha'], 'steps': steps,
                'restarts': restarts, 'random_start': True, 'loss': 'cross_entropy',
                'candidate_selection': 'success_first_all_iterates_including_clean',
                'eval_seed': cfg['eval_seed'], 'batch_size': cfg['batch_size']}
    label = f'pgd{steps}_r{restarts}'
    eval_dir = run_dir / 'evaluations' / label
    eval_dir.mkdir(parents=True, exist_ok=True)
    rows, examples = [], None
    begin = time.monotonic()
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.no_grad():
            pc = model(x).argmax(1)
        xf = fgsm(model, x, y, cfg['epsilon'])
        xp = pgd_linf(model, x, y, eps=cfg['epsilon'], alpha=cfg['eval_alpha'], steps=steps, restarts=restarts)
        with torch.no_grad():
            pf, pp = model(xf).argmax(1), model(xp).argmax(1)
        for j in range(len(y)):
            clean = int(pc[j].eq(y[j]).item())
            f = clean * int(pf[j].eq(y[j]).item())
            p = clean * int(pp[j].eq(y[j]).item())
            rows.append({'index': len(rows), 'label': y[j].item(), 'clean_correct': clean,
                         'fgsm_correct': f, 'pgd_correct': p, 'combined_correct': f * p})
        if examples is None:
            examples = tuple(t[:8].detach().cpu() for t in (x, xf, xp, y, pc, pf, pp))
        print(f'Evaluation {label}: {len(rows)}/{len(loader.dataset)}', flush=True)
    digest = hashlib.sha256(checkpoint.read_bytes()).hexdigest()
    group = hashlib.sha256(json.dumps({'config': cfg, 'protocol': protocol, 'synthetic': args.synthetic}, sort_keys=True).encode()).hexdigest()[:16]
    result = {'config_name': cfg['name'], 'protocol_id': group, 'mode': args.mode,
              'seed': args.seed, 'dataset': 'synthetic' if args.synthetic else 'MNIST',
              'n_test': len(rows), 'clean_accuracy': sum(r['clean_correct'] for r in rows) / len(rows),
              'fgsm_accuracy': sum(r['fgsm_correct'] for r in rows) / len(rows),
              'pgd_accuracy': sum(r['pgd_correct'] for r in rows) / len(rows),
              'combined_accuracy': sum(r['combined_correct'] for r in rows) / len(rows),
              'eval_steps': steps, 'eval_restarts': restarts, 'epsilon': cfg['epsilon'],
              'checkpoint_sha256': digest, 'eval_seconds': time.monotonic() - begin}
    state = torch.load(checkpoint, map_location='cpu', weights_only=False)
    result['selected_epoch'] = state['epoch']
    detail = {'metrics': result, 'protocol': protocol, 'config': cfg,
              'checkpoint': str(checkpoint), 'interpretation': 'Smoke/synthetic is plumbing only; empirical attack accuracy is not certified robustness.'}
    for folder in (eval_dir, run_dir):
        write_json(folder / 'evaluation.json', detail)
        write_csv(folder / 'metrics.csv', [result])
    write_csv(eval_dir / 'per_example.csv', rows)
    plot_examples(eval_dir / 'examples.png', examples, cfg['epsilon'])
    # Latest view for notebook convenience; all protocols remain under evaluations/.
    (run_dir / 'examples.png').write_bytes((eval_dir / 'examples.png').read_bytes())
    print(json.dumps(result, indent=2), flush=True)
    return result


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--config', required=True)
    parser.add_argument('--mode', choices=['baseline', 'adversarial'], default='baseline')
    parser.add_argument('--seed', type=int, default=0)
    parser.add_argument('--output-dir', default='runs')
    parser.add_argument('--data-dir', default='data')
    parser.add_argument('--device', default='auto', choices=['auto', 'cpu', 'cuda', 'mps'])
    parser.add_argument('--resume', action='store_true')
    parser.add_argument('--synthetic', action='store_true', help='Offline plumbing test only; never a scientific result.')
    group = parser.add_mutually_exclusive_group()
    group.add_argument('--train-only', action='store_true')
    group.add_argument('--eval-only', action='store_true')
    parser.add_argument('--checkpoint')
    parser.add_argument('--eval-steps', type=int)
    parser.add_argument('--eval-restarts', type=int)
    parser.add_argument('--epochs-per-call', type=int, default=0, help='0=all remaining; checkpointed chunks for free sessions.')
    args = parser.parse_args()
    cfg = json.loads(Path(args.config).read_text())
    if args.epochs_per_call < 0:
        parser.error('--epochs-per-call must be nonnegative')
    if args.synthetic and cfg['name'] != 'smoke':
        parser.error('--synthetic is restricted to the smoke configuration')
    if args.eval_only and not args.checkpoint:
        parser.error('--eval-only requires --checkpoint')
    if args.checkpoint and not args.eval_only:
        parser.error('--checkpoint is only for --eval-only; use --resume to continue training')
    seed_everything(args.seed)
    device = resolve_device(args.device)
    if device.type == 'cpu':
        torch.set_num_threads(min(torch.get_num_threads(), 4))
    run_dir = Path(args.output_dir) / (cfg['name'] + ('_synthetic' if args.synthetic else '')) / args.mode / f'seed_{args.seed}'
    run_dir.mkdir(parents=True, exist_ok=True)
    print(f'Device={device}; run={run_dir}; synthetic={args.synthetic}', flush=True)
    model = SmallCNN(width=cfg['width'], hidden=cfg['hidden']).to(device)
    loaders = make_loaders(args.data_dir, batch_size=cfg['batch_size'], seed=args.seed,
                           split_seed=cfg['split_seed'], val_size=cfg['val_size'],
                           train_limit=cfg['train_limit'], val_limit=cfg['val_limit'], test_limit=cfg['test_limit'],
                           num_workers=cfg['num_workers'], synthetic=args.synthetic)
    if not args.eval_only:
        # Validate an existing run before changing its provenance files.
        config_path = run_dir / 'config.json'
        if config_path.exists() and json.loads(config_path.read_text()) != cfg:
            raise ValueError('Existing run has a different config; choose a new output directory.')
        write_json(config_path, cfg)
        env = environment_info()
        env.update({'device_selected': str(device), 'seed': args.seed,
                    'split_seed': cfg['split_seed'], 'synthetic': args.synthetic,
                    'session_started_utc': datetime.now(timezone.utc).isoformat()})
        if not (run_dir / 'environment.json').exists():
            write_json(run_dir / 'environment.json', env)
        with (run_dir / 'sessions.jsonl').open('a') as f:
            f.write(json.dumps(env) + '\n')
        complete = train(model, loaders, device, cfg, args, run_dir)
        if not complete:
            print('Chunk complete. Resume with --resume; test evaluation waits until all epochs finish.')
            return
        if args.train_only:
            print('Training complete. Freeze choices using validation, then run --eval-only on best.pt.')
            return
    checkpoint = Path(args.checkpoint) if args.eval_only else run_dir / 'best.pt'
    state = torch.load(checkpoint, map_location='cpu', weights_only=False)
    if state['config'] != cfg or state['mode'] != args.mode or state['seed'] != args.seed or state['synthetic'] != args.synthetic:
        raise ValueError('Evaluation checkpoint does not match config/mode/seed/dataset.')
    model.load_state_dict(state['model'])
    evaluate(model, loaders['test'], device, cfg, args, run_dir, checkpoint)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile mnist_robust/model.py
"""Madry MNIST ağının ölçeklenebilir iki-convolution sürümü."""

import torch
from torch import nn


class SmallCNN(nn.Module):
    """Logits üretir; girişler normalize edilmemiş [0, 1] MNIST pikselleridir.

    width=32, hidden=1024 yazarların MNIST ağındaki katman boyutlarıdır.
    Daha küçük varsayılan ağ hızlı ilk deneyler için seçilmiştir.
    """

    def __init__(self, width: int = 16, hidden: int = 128):
        super().__init__()
        for name, value in (("width", width), ("hidden", hidden)):
            if isinstance(value, bool) or not isinstance(value, int) or value <= 0:
                raise ValueError(f"{name} pozitif bir tamsayı olmalı.")
        self.width = width
        self.hidden = hidden
        self.features = nn.Sequential(
            nn.Conv2d(1, width, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(width, 2 * width, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(7 * 7 * 2 * width, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 10),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))


In [ ]:
%%writefile mnist_robust/summarize.py
"""Aggregate identical protocols across independent training seeds (sample SD)."""
import argparse
import json
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

METRICS = ['clean_accuracy', 'fgsm_accuracy', 'pgd_accuracy', 'combined_accuracy']
KEYS = ['dataset', 'config_name', 'protocol_id', 'mode', 'n_test', 'eval_steps', 'eval_restarts', 'epsilon']


def summarize(runs, output, allow_single_seed=False):
    paths = sorted(Path(runs).glob('**/evaluations/*/metrics.csv'))
    if not paths:
        raise FileNotFoundError(f'No evaluations found in {runs}; run experiment first.')
    frames = []
    for path in paths:
        frame = pd.read_csv(path)
        frame['source'] = str(path)
        frames.append(frame)
    data = pd.concat(frames, ignore_index=True)
    if data.duplicated(KEYS + ['seed']).any():
        raise ValueError('Duplicate seed for an identical protocol. Use one run root per experiment.')
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    data.to_csv(output / 'per_seed.csv', index=False)
    rows = []
    for key, group in data.groupby(KEYS, dropna=False, sort=True):
        row = dict(zip(KEYS, key))
        row['n_seeds'] = len(group)
        row['seeds'] = ','.join(map(str, sorted(group.seed.tolist())))
        row['status'] = ('SMOKE_ONLY' if row['config_name'] == 'smoke' or row['dataset'] == 'synthetic'
                         else ('INCOMPLETE_SINGLE_SEED' if len(group) < 2 else 'MEASURED'))
        for metric in METRICS:
            row[metric + '_mean'] = group[metric].mean()
            row[metric + '_std'] = group[metric].std(ddof=1)
        rows.append(row)
    summary = pd.DataFrame(rows)
    summary.to_csv(output / 'summary.csv', index=False)
    lines = ['# Measured outputs', '', 'Accuracies are percentages below; CSV stores fractions. '
             '± is sample standard deviation across training seeds (ddof=1), not a confidence interval. '
             'One seed has undefined SD. SMOKE_ONLY is a pipeline check, never evidence of reproduction.', '',
             '| Dataset/config/protocol | Mode | Seeds | Clean | FGSM | PGD | Combined | Status |',
             '|---|---|---|---|---|---|---|---|']
    for row in rows:
        def fmt(metric):
            mean, sd = row[metric+'_mean'], row[metric+'_std']
            return f'{100*mean:.2f} ± {100*sd:.2f}' if pd.notna(sd) else f'{100*mean:.2f} (SD undefined)'
        values = [fmt(metric) for metric in METRICS]
        lines.append(f"| {row['dataset']}/{row['config_name']}/{row['protocol_id']} | {row['mode']} | {row['seeds']} | " + ' | '.join(values) + f" | {row['status']} |")
    lines += ['', 'Combined accuracy survives clean + FGSM + every retained PGD candidate. '
              'PGD accuracy is an empirical result for the stated attack; it does not certify robustness.']
    (output / 'summary.md').write_text('\n'.join(lines) + '\n')
    groups = list(summary.groupby(['dataset', 'config_name', 'protocol_id'], sort=True))
    fig, axes = plt.subplots(len(groups), 1, figsize=(9, 4.2*len(groups)), squeeze=False)
    for ax, ((dataset, config, protocol), group) in zip(axes[:, 0], groups):
        x = np.arange(len(METRICS))
        width = 0.75 / len(group)
        for i, (_, row) in enumerate(group.iterrows()):
            means = [100*row[m+'_mean'] for m in METRICS]
            sd = np.array([100*row[m+'_std'] for m in METRICS])
            # Do not draw a zero-width error bar for an undefined single-seed SD.
            ax.bar(x + (i - (len(group)-1)/2)*width, means, width,
                   yerr=sd if np.isfinite(sd).all() else None, capsize=4,
                   label=f"{row['mode']} (n={row['n_seeds']})")
        ax.set_xticks(x, ['Clean', 'FGSM', 'PGD', 'Combined'])
        ax.set_ylim(0, 100)
        ax.set_ylabel('Accuracy (%)')
        suffix = ' — PIPELINE CHECK ONLY' if config == 'smoke' or dataset == 'synthetic' else ''
        ax.set_title(f'{dataset} / {config}: PGD-{int(group.iloc[0].eval_steps)} × {int(group.iloc[0].eval_restarts)}{suffix}')
        ax.legend()
    fig.tight_layout()
    fig.savefig(output / 'accuracy.png', dpi=180)
    plt.close(fig)
    histories = sorted(Path(runs).rglob('history.csv'))
    if histories:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        for path in histories:
            h = pd.read_csv(path)
            name = '/'.join(path.parts[-4:-1])
            axes[0].plot(h.epoch, h.val_clean_accuracy*100, label=name)
            axes[1].plot(h.epoch, h.val_pgd_accuracy*100, label=name)
        for ax, title in zip(axes, ['Validation clean', 'Validation PGD (configuration-specific attack)']):
            ax.set(title=title, xlabel='Epoch', ylabel='Accuracy (%)', ylim=(0, 100))
            ax.legend(fontsize=7)
        fig.tight_layout()
        fig.savefig(output / 'training_curves.png', dpi=180)
        plt.close(fig)
    print('\n'.join(lines))
    if not allow_single_seed and (summary.n_seeds < 2).any():
        raise SystemExit('Outputs saved; at least one group has one seed. Add seeds, or use --allow-single-seed for smoke/pilot inspection.')
    return summary


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--runs', default='runs')
    parser.add_argument('--output', default='results')
    parser.add_argument('--allow-single-seed', action='store_true')
    args = parser.parse_args()
    summarize(args.runs, args.output, args.allow_single_seed)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile mnist_robust/utils.py
"""Seed ve ortam kaydı: sonuçlar kullanılan koşullarla birlikte saklanır."""

from importlib.metadata import PackageNotFoundError, version
import os
import platform
import random
import sys

import numpy as np
import torch


def seed_everything(seed: int):
    """Aynı ortamda tekrar çalıştırılabilirlik; farklı donanımlar eşitlik garantisi vermez."""
    if isinstance(seed, bool) or not isinstance(seed, int) or not 0 <= seed < 2**32:
        raise ValueError("seed, [0, 2**32) aralığında bir tamsayı olmalı.")
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    # Bazı cihazlarda deterministik implementasyonu olmayan operatörleri bildir.
    torch.use_deterministic_algorithms(True, warn_only=True)


def resolve_device(requested="auto"):
    """İstenen cihaz yoksa sessizce fallback yapmak yerine anlaşılır hata verir."""
    if requested not in {"auto", "cpu", "cuda", "mps"}:
        raise ValueError("device auto, cpu, cuda veya mps olmalı.")
    has_mps = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    if requested == "auto":
        requested = "cuda" if torch.cuda.is_available() else "mps" if has_mps else "cpu"
    if requested == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA bulunamadı. Colab/Kaggle GPU'yu etkinleştirin veya --device cpu kullanın.")
    if requested == "mps" and not has_mps:
        raise RuntimeError("Bu ortamda MPS kullanılamıyor; --device cpu kullanın.")
    return torch.device(requested)


def environment_info():
    """JSON'a yazılabilir temel sürüm/donanım bilgileri; torchvision import edilmez."""
    try:
        torchvision_version = version("torchvision")
    except PackageNotFoundError:
        torchvision_version = None
    return {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "torch": str(torch.__version__),
        "torchvision": torchvision_version,
        "numpy": np.__version__,
        "cuda_runtime": torch.version.cuda,
        "cudnn": torch.backends.cudnn.version(),
        "cuda_available": torch.cuda.is_available(),
        "cuda_devices": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
        "mps_available": bool(hasattr(torch.backends, "mps") and torch.backends.mps.is_available()),
        "deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),
        "deterministic_warn_only": torch.is_deterministic_algorithms_warn_only_enabled(),
    }


In [ ]:
%%writefile configs/main.json
{
  "name": "main", "epochs": 10, "batch_size": 128, "lr": 0.001,
  "width": 16, "hidden": 128, "split_seed": 2026, "val_size": 5000,
  "train_limit": null, "val_limit": null, "test_limit": null, "num_workers": 0,
  "epsilon": 0.3, "train_alpha": 0.01, "train_steps": 40,
  "val_steps": 40, "val_restarts": 1,
  "eval_alpha": 0.01, "eval_steps": 100, "eval_restarts": 10, "eval_seed": 12345
}


In [ ]:
%%writefile configs/pilot.json
{
  "name": "pilot", "epochs": 3, "batch_size": 128, "lr": 0.001,
  "width": 16, "hidden": 128, "split_seed": 2026, "val_size": 5000,
  "train_limit": 10000, "val_limit": 1000, "test_limit": 1000, "num_workers": 0,
  "epsilon": 0.3, "train_alpha": 0.01, "train_steps": 10,
  "val_steps": 10, "val_restarts": 1,
  "eval_alpha": 0.01, "eval_steps": 40, "eval_restarts": 3, "eval_seed": 12345
}


In [ ]:
%%writefile configs/smoke.json
{
  "name": "smoke", "epochs": 1, "batch_size": 64, "lr": 0.001,
  "width": 8, "hidden": 32, "split_seed": 2026, "val_size": 5000,
  "train_limit": 256, "val_limit": 64, "test_limit": 64, "num_workers": 0,
  "epsilon": 0.3, "train_alpha": 0.01, "train_steps": 2,
  "val_steps": 3, "val_restarts": 1,
  "eval_alpha": 0.01, "eval_steps": 5, "eval_restarts": 2, "eval_seed": 12345
}


In [ ]:
%%writefile requirements.txt
# Keep the platform's compatible CUDA torch/torchvision pair when already installed.
# Exact versions and hardware are saved into each run's environment.json.
torch>=2.5,<3
torchvision>=0.20,<1
numpy>=1.26,<3
matplotlib>=3.8,<4
pandas>=2.2,<4


In [ ]:
%%writefile README.md
# MNIST üzerinde PGD ile dayanıklı öğrenme

CMP4509 Deep Learning grup projesi: Madry ve arkadaşlarının **“Towards Deep Learning Models Resistant to Adversarial Attacks”** çalışmasının ana fikrini, ücretsiz Colab/Kaggle GPU'larında çalışabilecek küçük bir PyTorch deneyiyle sınamak.

**Araştırma sorumuz:** Aynı ağı normal eğitimle ve PGD adversarial training ile eğittiğimizde, temiz görüntü doğruluğu ve güçlü PGD saldırısı altındaki doğruluk nasıl değişiyor?

Bu depo bir başlangıç uygulamasıdır. Eğitim yapılmadan sonuç uydurulmaz. Küçük ağ, eğitim bütçesi ve doğrulama bölümü nedeniyle **makaledeki sayıları birebir tekrar ettiğimizi iddia etmiyoruz**. Amaç, ana sonucu kontrollü ve tekrar edilebilir biçimde incelemek.

## İlk oturum: küçük bir deneyi uçtan uca bitirelim

1. `notebooks/01_mnist_reproduction.ipynb` dosyasını Colab'a yükle veya Kaggle Notebook olarak içe aktar. Notebook bütün kaynak dosyalarını görünür hücrelerde içerir; başka bir depoyu klonlamaya gerek yoktur.
2. Colab'da **Runtime → Change runtime type → GPU**, Kaggle'da **Settings → Accelerator → GPU** seç. Kaggle'da MNIST indirmek için Internet açık olmalı. GPU kotası ve erişim değişebilir.
3. Varsayılan `CONFIG_NAME = "smoke"` ayarını koru ve hücreleri yukarıdan aşağı çalıştır. Bu küçük koşu önce normal eğitimi, sonra PGD eğitimini ve değerlendirmeyi sınar. CPU'da da çalışabilir.
4. İlk sonuç tablosunu ve temiz/saldırılmış örnekleri kontrol et. **Ayarlar, kullanılan cihaz ve sonuç tablosunun ekran görüntüsünü al.** Bunlar ileride sunumun deney sürecini anlatacak.
5. Notebook'un sonundaki arşivi indir. Colab oturumu kapanınca geçici dosyalar silinebilir; uzun koşudan önce Drive kaydını aç. Kaggle'da **Save Version** ile çıktılarını sakla.

Smoke test bilimsel sonuç değildir: 256 eğitim örneği ve tek epoch yalnızca veri, gradyan, saldırı, kayıt ve raporlama zincirini kontrol eder.

Min–max, FGSM türetimi, PGD güncellemesi ve iki farklı gradyanı adım adım anlamak için [kısa teori rehberini](docs/THEORY.md) oku. Rapor için [BibTeX kaynakları](docs/references.bib) da hazır.

## Fikir: saldırganın bulduğu zor örneklerle öğrenmek

Makalenin hedefi şudur:

\[
\min_{\theta}\;\mathbb{E}_{(x,y)}\left[\max_{\|\delta\|_{\infty}\leq\epsilon}\mathcal{L}(f_\theta(x+\delta),y)\right].
\]

İçteki **max**, mevcut modeli zorlayacak izinli bir piksel değişimini arar. Dıştaki **min**, model ağırlıklarını bu zor örneklerdeki kaybı azaltacak yönde günceller. PGD, içteki aramaya pratik bir yaklaşım sağlar; gerçek en kötü saldırıyı bulduğunu garanti etmez.

- MNIST piksel aralığı `[0, 1]`; tehdit sınırı **L∞, ε = 0.3**. Görüntülere ayrıca ortalama/standart sapma normalizasyonu uygulanmaz, böylece epsilon doğrudan bu piksel ölçeğindedir.
- **FGSM:** kaybı büyüten yönde tek adım.
- **PGD:** küçük adımlar, her adımda orijinal görüntünün izinli ε komşuluğuna ve `[0, 1]` aralığına geri yansıtma.
- **Rastgele başlangıç:** her restart saldırıyı izinli bölgenin başka bir noktasından başlatır.
- **Güçlü değerlendirme:** eğitimde kullanılandan daha fazla PGD adımı ve birden fazla restart. Bir örneği herhangi bir restart/adım bozabiliyorsa değerlendirmede bu başarısızlık korunur; restart doğruluklarının ortalaması alınmaz.

Ölçtüğümüz dayanıklılık, seçilen saldırı ve bütçe altındaki ampirik doğruluktur; matematiksel bir dayanıklılık sertifikası değildir.

## Resmî uygulamadan aldıklarımız ve değiştirdiklerimiz

Kaynaklar: [makale](https://arxiv.org/abs/1706.06083), [yazarların MNIST Challenge uygulaması](https://github.com/MadryLab/mnist_challenge), [resmî ayarlar](https://github.com/MadryLab/mnist_challenge/blob/master/config.json), [resmî model](https://github.com/MadryLab/mnist_challenge/blob/master/model.py), [resmî PGD saldırısı](https://github.com/MadryLab/mnist_challenge/blob/master/pgd_attack.py).

| Seçim | Yazarların MNIST uygulaması | Bu projenin `main` başlangıç ayarı |
|---|---|---|
| Piksel ölçeği / ε | `[0,1]` / `0.3` | Aynı |
| Eğitim saldırısı | PGD, 40 adım, adım boyu `0.01`, rastgele başlangıç, cross-entropy | Aynı saldırı ailesi ve değerler |
| Ağ | 5×5 convolution: 32 ve 64 kanal; max-pooling; 1024 birimli gizli katman | Aynı 2-convolution düzeni; 16 ve 32 kanal; 128 birimli gizli katman |
| Optimizasyon | Adam, `1e-4`, batch 50, 100.000 güncelleme | Adam, `1e-3`, batch 128, 10 epoch |
| Veri kullanımı | Resmî kodun eğitim akışı | 55.000 eğitim / sabit 5.000 doğrulama / 10.000 test |
| Başlatma | Resmî TensorFlow ağırlık/bias başlatması | PyTorch varsayılan başlatması |
| Değerlendirme | Makalede ve challenge'da farklı saldırı bütçeleri raporlanır | `main`: PGD 100 × 10 restart; dondurulmuş final: 100 × 20 restart |

Resmî konfigürasyondaki PGD 40 ayarı tek başına yeterli bir final değerlendirme standardı sayılmaz. Makalenin güçlü saldırı değerlendirmesini örnek alarak eğitimden güçlü bir test kullanıyoruz. Daha büyük ağ/uzun eğitim sonradan deney olabilir; önce küçük koşuyu gerçekten tamamlayalım.

## Dosyalar

```text
configs/                  smoke, pilot ve main JSON ayarları
mnist_robust/              veri, model, saldırılar, eğitim, değerlendirme ve özetleme
notebooks/
  01_mnist_reproduction.ipynb  kendi kaynaklarını içeren, baştan sona notebook
tests/                    saldırı sınırları ve ilgili doğrulamalar
docs/
  THEORY.md                min–max, FGSM/PGD ve metriklerin açıklaması
  references.bib           birincil kaynaklar
  EXPERIMENT_PLAN.md       deney sırası, ölçümler ve yorumlama
  COURSE_DELIVERABLES.md   ders teslimleri ve ekran görüntüsü kontrol listesi
runs/                     çalışınca oluşur: checkpoint, geçmiş, ayarlar, metrikler
results/                  özet CSV, tablolar ve şekiller
```

## Yerel kullanım veya terminalden çalışma

Yeni bir Python ortamında, projenin kök klasöründen:

```bash
python -m pip install -r requirements.txt
python -m mnist_robust.experiment --config configs/smoke.json --mode baseline --seed 0 --output-dir runs --device auto
python -m mnist_robust.experiment --config configs/smoke.json --mode adversarial --seed 0 --output-dir runs --device auto
python -m mnist_robust.summarize --runs runs --output results --allow-single-seed
```

Colab/Kaggle'ın önceden kurulu GPU destekli PyTorch paketini gereksiz yere değiştirme. Notebook yalnızca gerekli eksik paketleri kurar; uyumsuz bir kurulumda açık hata verir. Aynı sonuçları karşılaştırırken kaydedilmiş Python/PyTorch/CUDA ortam bilgisini de sakla.

İnternet veya MNIST erişimi yoksa yalnızca kod yolunu kontrol etmek için `--synthetic` eklenebilir. Sentetik veri koşuları MNIST sonucu değildir ve gerçek veriyle birleştirilmez.

## Deneyi büyütme sırası

| Ayar | Eğitim / doğrulama / test örneği | Epoch | Eğitim PGD | Test PGD | Kullanım |
|---|---:|---:|---:|---:|---|
| `smoke` | 256 / 64 / 64 | 1 | 2 | 5 × 2 restart | Kodun çalıştığını doğrula |
| `pilot` | 10.000 / 1.000 / 1.000 | 3 | 10 | 40 × 3 restart | GPU maliyetini ve öğrenmeyi gözle |
| `main` | 55.000 / 5.000 / 10.000 | 10 | 40 | 100 × 10 restart | Üç seed ile asıl karşılaştırma |
| Dondurulmuş final | `main` test kümesi | Yeni eğitim yok | — | 100 × 20 restart | Seçilen checkpoint'leri güçlü test et |

Bunlar ücretsiz GPU için **ölçeklenebilir başlangıç bütçeleridir**; verilen sürede tamamlanacakları garanti değildir. `main` PGD eğitimi ve özellikle 20 restart test pahalıdır. Önce pilot süresini ölç, sonra süre/örnek sayısını kaydet. Bütçeyi değiştirirsen yeni ayarı ayrı isimle sakla ve hangi protokolün kullanıldığını rapora yaz.

Model ve eğitim ayarlarını **doğrulama kümesiyle** seç. Test setini görmek için ayarları dondurmayı bekle. Pilotun test sonuçlarına bakarak ana modeli ayarlama; pilot koşularında `--train-only` kullan:

```bash
python -m mnist_robust.experiment --config configs/pilot.json --mode baseline --seed 0 --output-dir runs --device auto --train-only
python -m mnist_robust.experiment --config configs/pilot.json --mode adversarial --seed 0 --output-dir runs --device auto --train-only
```

Aynı düzeni `main` için **seed 0, 1, 2** ve iki eğitim türünde çalıştır. Notebook bunu döngüyle yapar. Normal eğitimde en iyi temiz doğrulama checkpoint'i, PGD eğitiminde en iyi saldırılı doğrulama checkpoint'i seçilir; karşılaştırmada bu seçim kuralını açıkça belirt.

Kesilen bir eğitim aynı komuta `--resume` eklenerek son tamamlanmış checkpoint'ten sürdürülebilir. Oturumu kısa parçalara bölmek için `--epochs-per-call 1` kullanabilirsin; bütün epoch'lar bitmeden otomatik test başlamaz. Veri, model ve eğitim ayarlarını değiştirdikten sonra eski checkpoint'e devam etme; ayrı koşu başlat.

Ayarları dondurduktan sonra örneğin PGD eğitimli seed 0 modelini güçlü test et:

```bash
python -m mnist_robust.experiment --config configs/main.json --mode adversarial --seed 0 --output-dir runs --device auto --eval-only --checkpoint runs/main/adversarial/seed_0/best.pt --eval-steps 100 --eval-restarts 20
```

Aynı testi `baseline` ve diğer seed'ler için de uygula. Sonrasında:

```bash
python -m mnist_robust.summarize --runs runs/main --output results/main
```

Özetleyici farklı veri/config/saldırı protokollerini ayrı tutar. `smoke`, `pilot`, `main` veya farklı restart sayıları aynı ortalamaya girmemelidir. Tek seed ile örnek çıktı görmek için `--allow-single-seed` vardır; tek seed için standart sapma güvenilir biçimde hesaplanamaz. Ana raporda **her model için üç seed**, seed başına değerler ve **ortalama ± örnek standart sapma** bulunmalı.

## Neler kaydediliyor?

Her koşu `runs/{config}/{mode}/seed_{seed}/` altında ayarları, ortam bilgisini, `last.pt`, `best.pt`, eğitim geçmişini ve test yapıldıysa metrikleri/örnek görselini saklar. Sentetik smoke klasöründe `smoke_synthetic` adı kullanılır. Farklı test bütçeleri `evaluations/pgd{steps}_r{restarts}/` altında ayrı korunur; koşu kökündeki dosyalar son değerlendirmeyi gösterir. `last.pt` devam etmek, `best.pt` değerlendirme için kullanılır. Toplu özetleyici `per_seed.csv`, `summary.csv`, `summary.md`, `accuracy.png`, `training_curves.png` üretir.

Temiz, FGSM ve PGD doğruluklarını birlikte incele. Yalnız temiz doğruluk, saldırıya dayanıklılığı göstermez. PGD adım/restart sayısını artırınca dayanıklılık belirgin düşüyorsa zayıf test fazla iyimser sonuç vermiştir. Sadece başarılı görünen seed'i seçme; bütün önceden belirlenmiş seed'leri raporla.

**Şu an ölçülmüş bilimsel sonuç yok.** Çalıştırılan smoke testler yalnızca çalışma doğrulamasıdır. Kullanıcı GPU eğitimini yaptıktan sonra asıl tablolar, şekiller ve rapor bulguları üretilecek.

## Ders teslimleri

- **Public GitHub deposu:** kaynaklar, bağımsız notebook, README, deney ayarları ve yeniden üretim adımları. Büyük checkpoint/veri dosyalarını Git geçmişine koyma; paylaşılan çıktılar için ayrıca bağlantı kullan.
- **Rapor:** Teams'teki `CMP4509_report_template.zip` kullanılarak hazırlanmış PDF; **13 Aralık 2026, 23:59 (Türkiye saati)**. Rapor içinde public depo bağlantısı bulunmalı.
- **Sunum:** **30 Kasım veya 7 Aralık 2026**, soru-cevap dahil **10 dakika**, hedef **8–10 slayt**. Her grup üyesi bütün projeyi açıklayabilmeli.

Ders şablonu bu çalışma alanında henüz yok; şablon gelmeden ders formatında tamamlanmış rapor varmış gibi davranılmayacak. Ayrıntılar için [deney planı](docs/EXPERIMENT_PLAN.md) ve [teslim kontrol listesi](docs/COURSE_DELIVERABLES.md).

Yerel kontrollerin kapsamı ve henüz çalıştırılmayan deneyler: [doğrulama kaydı](docs/VALIDATION.md).

Kaynak dosyalarını değiştirdikten sonra bağımsız notebook’u eşitle: `python tools/build_notebook.py`. Gerçek, yalnız kontrol amaçlı çıktılar [smoke_check](results/smoke_check/README.md) altında; bilimsel sonuç değildir.


In [ ]:
%%writefile docs/EXPERIMENT_PLAN.md
# Deney planı: önce çalışan zincir, sonra anlamlı karşılaştırma

## 1. Smoke test: bağlantılar doğru mu?

Notebook'un varsayılan `smoke` ayarını iki eğitim türünde çalıştır. Bu koşuda amaç iyi doğruluk almak değildir. Şunları görmeliyiz:

- MNIST indiriliyor ve tensör değerleri `[0,1]` aralığında.
- İki convolution katmanlı ağ ileri/geri geçiş yapıyor.
- FGSM ve PGD görüntüyü `[0,1]` aralığında ve orijinale L∞ uzaklığı `ε≤0.3` olacak şekilde tutuyor.
- Eğitim kaybı, doğrulama, checkpoint, CSV ve görseller üretiliyor.
- Daha fazla restart altında başarıyla saldırılmış bir örnek yanlışlıkla tekrar doğru sayılmıyor.

**Ekran görüntüsü:** GPU/CPU bilgisi, tüm smoke hücrelerinin tamamlandığı ekran, temiz ve saldırılmış görüntü örnekleri. Bunlar “deney ortamı çalışıyor” kanıtıdır; asıl başarı sonucu değildir.

## 2. Pilot: öğrenme ve süreyi gözle

`pilot`, önce `baseline`, sonra `adversarial`; başlangıçta seed 0. `--train-only` kullan. Normal eğitim sadece temiz görüntüyü görür. Adversarial eğitim her batch için mevcut modelle PGD örnekleri üretir ve o örneklerde güncelleme yapar.

Aynı veri bölümü ve aynı ağ, iki eğitim yönteminin karşılaştırılmasını kolaylaştırır. Eşit epoch sayısı eşit hesaplama maliyeti değildir: PGD eğitimi birçok ek gradyan hesabı yapar. Süreyi de raporlamak bu farkı görünür kılar.

Kontrol et: kayıp sonlu mu, temiz doğrulama öğreniyor mu, PGD doğrulama nasıl değişiyor, bir epoch ne kadar sürüyor? Hiperparametre kararlarını burada ve doğrulama kümesinde ver. Test setini model seçimi için kullanma.

**Ekran görüntüsü:** iki eğitim türünün öğrenme eğrileri, kullanılan config ve süre. Bir hata varsa hata mesajını ve çözümünü deney günlüğüne yaz.

## 3. Ana deney: karşılaştırmayı önceden sabitle

Pilot sonrası ağ genişliği, epoch, optimizer, öğrenme oranı, eğitim PGD bütçesi ve checkpoint seçim kuralını dondur. Varsayılan `main` ayarları nihai karar yerine ilk bütçe önerisidir. Değişiklikleri yeni config dosyasında sakla.

Her iki eğitim yöntemi için seed kümesi **[0,1,2]**. Aynı seed numaralarını eşleştir ve bütün koşuları sakla. Eğitim/doğrulama bölümü sabit kalır; seed ağırlık başlangıcı, batch sırası ve saldırı rastgeleliğini etkiler. Donanım veya yazılım değişimleri tam bit düzeyinde eşitliği etkileyebilir; ortam kaydını karşılaştır.

Normal eğitim checkpoint'i temiz doğrulama doğruluğuyla; adversarial eğitim checkpoint'i saldırılı doğrulama doğruluğuyla seçilir. Bu farklılık raporda açık olmalı. Eğitim sonunda `last.pt` mevcut olsa da test için önceden belirlenen `best.pt` kullanılır.

Uzun koşuda saklama konumunu önce ayarla. Colab için Drive, Kaggle için `/kaggle/working` ve Save Version kullan. Bir epoch bitmeden oturum kesilirse o epoch yeniden çalışabilir.

## 4. Dondurulmuş test: saldırıyı güçlendir

Bütün kararlar verildikten sonra her checkpoint için:

| Ölçüm | Amaç |
|---|---|
| Temiz test doğruluğu | Normal görüntülerde performans |
| FGSM, ε=0.3 | Tek adımlı saldırıyla karşılaştırma |
| PGD, ε=0.3, α=0.01, 100 adım, 20 restart | Eğitimden güçlü beyaz-kutu dayanıklılık testi |
| Koşu ve değerlendirme süresi | Ücretsiz GPU bütçesini görünür kılma |

PGD için modelin gradyanı kullanılabilir; saldırı **beyaz-kutu** ve **hedefsizdir**. Her restart kaybı büyütmeye çalışır. Örneğin herhangi bir başlangıç/adım/restart ile yanlış sınıflandırılması başarısızlık sayılır. Sıfır değişim de tehdit kümesinde olduğundan temizken yanlış olan örnek dayanıklı kabul edilmez.

Finalden önce küçük doğrulama altkümesinde 40/100 adım ve 1/10/20 restart bütçelerini karşılaştırabilirsin. Bu bir saldırı kontrolüdür; test sonuçlarına göre ağı yeniden ayarlama. Birden çok saldırı bütçesini raporlarsan hangi checkpoint ve örnek kümesini kullandığını açık yaz. Daha güçlü saldırı bütçesinin tek rastgele çalışmada mutlaka daha düşük sayı üretmesi garanti değildir; ortak başlangıçlar ve birikimli başarısızlık takibi bu kontrolü daha anlamlı kılar.

GPU süresi yetmezse sonucu gizleme: daha küçük bütçeyi açıkça yaz, bütün modellerde aynı protokolü kullan ve güçlü testin eksik kaldığını sınırlılık olarak belirt. Tam veri testi yapılmış gibi konuşma.

## 5. Birleştirme ve yorum

Özet CSV'de seed başına temiz/FGSM/PGD doğrulukları; toplu tabloda model türü, seed sayısı, ortalama ve **örnek standart sapma (ddof=1)** yer alır. Tek seed için standart sapma sıfırmış gibi sunulmaz. Yüzde değer ile yüzde puan farkını ayır: iki modelin doğruluk farkı yüzde puandır.

Farklı config, veri kaynağı, test örnek sayısı, saldırı adımı veya restart bütçelerini tek ortalamada karıştırma. `smoke` ve `pilot` ana deneyin satır sayısını artırmaz. Üç seed belirsizliği göstermek için mütevazı bir başlangıçtır; güçlü istatistiksel genelleme iddiası kurma.

Raporun merkezindeki grafik aynı protokol altında **normal eğitim / PGD eğitimi × temiz / FGSM / güçlü PGD** karşılaştırmasıdır. İkinci şekil öğrenme eğrileri; üçüncü şekil birkaç aynı örneğin temiz/saldırılmış görünümü olabilir. Örnekler tek başına nicel sonuç yerine geçmez.

Şu sorulara sonuçlardan cevap ver:

1. PGD eğitimi güçlü saldırı altındaki doğruluğu artırdı mı?
2. Temiz doğrulukta ve hesaplama süresinde ne değişti?
3. Sonuç seed'ler arasında tutarlı mı?
4. Daha güçlü test önceki izlenimimizi değiştirdi mi?
5. Makaledeki kurulumdan hangi farklar sonuçları açıklayabilir?

Beklenen yön bir hipotezdir. Sonuç beklediğimiz gibi çıkmazsa kod, saldırı bütçesi ve optimizasyonu inceleyip eldeki sayıları dürüstçe raporlarız.

## Deney günlüğü şablonu

Her önemli koşu için bu alanları bir Markdown dosyasına veya paylaşılan grup günlüğüne ekle:

```text
Tarih / çalıştıran kişi:
Git commit veya kullanılan kaynak arşivi:
Config adı ve değişiklikler:
Seed / model türü:
GPU ve ortam:
Checkpoint yolu:
Epoch / eğitim süresi:
Doğrulama gözlemi:
Test açıldı mı? Ayarlar donduruldu mu?
Saldırı bütçesi / test örnek sayısı:
Çıktı arşivi ve ekran görüntüsü:
Sonraki karar ve gerekçesi:
```


In [ ]:
%%writefile docs/COURSE_DELIVERABLES.md
# CMP4509 teslim kontrol listesi

Bu belge ders teslimlerini ve deneylerden toplanacak kanıtları izler. Tarihler Türkiye saatine göredir. Paper sign-up tamamlandı.

| Teslim | Tarih / şart | Şimdiki durum |
|---|---|---|
| Public GitHub deposu | Kaynak + tekrar üretim talimatları; raporda bağlantısı olmalı | Yerel başlangıç hazırlanıyor; yayınlanma durumunu gerçek GitHub bağlantısıyla doğrula |
| Bağımsız notebook | Temiz Colab/Kaggle oturumunda yukarıdan aşağı çalışmalı | Varsayılan smoke; gerçek GPU sonuçları kullanıcı tarafından çalıştırılacak |
| Sunum | 30 Kasım **veya** 7 Aralık 2026; QA dahil 10 dakika | Grubun kesin tarihini ders duyurusundan netleştir |
| Rapor PDF | **13 Aralık 2026, 23:59**; Teams üzerinden | Şablon ve bilimsel deney sonuçları bekleniyor |
| Ders şablonu | Teams: `CMP4509_report_template.zip` | Bu çalışma alanında henüz bulunmuyor |

## Önerilen sıra

1. **İlk çalışma oturumu:** smoke test; kod ve saldırı mekanizmasını birlikte anlama; çalışan notebook ekran görüntüsü.
2. **Sonraki GPU oturumu:** pilot normal eğitim ve pilot PGD eğitimi; süre ölçümü; doğrulama eğrileri; hata notları.
3. **Ana deney öncesi:** ayarları ve seed listesini dondur; grup üyelerine koşuları dağıt; kalıcı kayıt yerini doğrula.
4. **Ana deney:** her iki model için üç seed; bütün checkpoint, config, geçmiş ve ortam dosyalarını sakla.
5. **Güçlü test:** aynı PGD 100 adım × 20 restart protokolü; gerçek CSV ve şekiller; tabloyu bağımsız olarak kontrol et.
6. **Sunumdan önce:** 8–10 slayt; yaklaşık 7–8 dakika anlatım + 2–3 dakika sorular için prova. Her üye tüm akışı açıklasın.
7. **Rapor:** gerçek ders LaTeX şablonunu kullan; public depo bağlantısını ekle; PDF'yi görsel olarak kontrol et; Teams teslimini doğrula.

## Public GitHub deposu

Depo görünürlüğü gerçekten **public** olmalı; bağlantıyı oturum açılmamış bir pencerede açarak doğrula. Şunlar depoda yer alsın:

- README ve çalıştırma sırası.
- Bütün kaynak kodu, requirements ve isimlendirilmiş config dosyaları.
- Kaynaklarını içeren notebook; tek bir Colab oturumuna veya özel Drive yoluna bağımlı olmamalı.
- Gerçek sonuç tablosu/şekilleri, kullanılan seed'ler ve saldırı bütçesi.
- Makale ve resmî yazar uygulamasına atıf; birebir tekrar olmayan noktaların açıklaması.

MNIST verisi, büyük checkpoint'ler, sanal ortamlar ve erişim anahtarları repoya eklenmesin. Checkpoint'ler için paylaşılabilir ayrı bir çıktı bağlantısı kullanılabilir. Public depo açıldığında README'deki yayın bağlantısını ve rapordaki URL'yi güncelle; bir repo URL'sini var olmadan yazma.

## Rapor iskeleti: şablona aktarılacak içerik

Bu bir içerik planıdır; dersin biçim şablonu yerine geçmez.

1. Problem ve motivasyon: temiz doğruluk neden tek başına yeterli değil?
2. Madry çalışması: min–max hedefi, L∞ tehdit modeli, PGD ile iç optimizasyon.
3. Yöntem: veri bölümü, iki convolution ağı, baseline/PGD eğitimi, optimizer, bütçe, seed ve checkpoint seçimi.
4. Değerlendirme: temiz/FGSM/PGD, rastgele başlangıçlar, restart başarısızlığını birleştirme, testin dondurulması.
5. Gerçek sonuçlar: seed başına tablo, ortalama ± std, süre, grafikler ve örnekler.
6. Tartışma: temiz doğruluk–dayanıklılık ilişkisi, güçlü testin etkisi, makaleden farklar ve sınırlılıklar.
7. Sonuç, katkı dağılımı, kaynakça ve public depo bağlantısı.

Sonuç gelmeden sayı, “başarılı tekrar” iddiası veya nihai sonuç paragrafı doldurma. Makaleden alınan sayı varsa açıkça **literatür sonucu** olarak etiketle ve kendi tablonla karıştırma.

## Sunumda herkesin açıklayabilmesi gerekenler

- İçteki max ve dıştaki min ne yapıyor?
- ε=0.3 hangi piksel ölçeğinde? L∞ sınırı ne demek?
- FGSM ile PGD arasındaki fark ne?
- Rastgele başlangıç/restart neden gerekli?
- Eğitimden daha güçlü test neden kullanılıyor?
- Hangi ayarlar makaleden farklı ve neden?
- Veri sızıntısı olmadan checkpoint nasıl seçildi?
- Sonuç üç seed'de ne kadar değişti? Hesaplama maliyeti neydi?

## Ekran görüntüsü ve çıktı listesi

- [ ] Ortam/GPU ve config hücresi.
- [ ] Smoke testin tamamlanması.
- [ ] Baseline ve PGD eğitim eğrileri.
- [ ] Aynı görüntünün temiz/FGSM/PGD örneği ve ε kontrolü.
- [ ] Seed başına gerçek sonuç CSV tablosu.
- [ ] Ortalama ± std karşılaştırma grafiği.
- [ ] Public GitHub ana sayfası ve notebook.
- [ ] Doğru şablonla üretilmiş son PDF.
- [ ] Teams teslim onayı.

Ekran görüntüsü sayısal dosyaların yerine geçmez. CSV, config ve kaynak sürümü temel kayıttır; görüntüler anlatımı destekler. Bu belge bir kontrol listesidir; otomatik hatırlatıcı kurulmuş değildir.


In [ ]:
%%writefile docs/THEORY.md
# Kodu anlayarak çalıştırmak: min–max, FGSM ve PGD

Elimizde bir rakam görüntüsü `x`, doğru etiket `y` ve ağırlıkları `θ` olan model `fθ` var. Normal eğitim, temiz görüntülerin cross-entropy kaybını azaltır. Adversarial eğitim, modelin küçük ama kasıtlı değişikliklere karşı da doğru kalmasını hedefler.

## 1. Min–max hedefini okuyalım

\[
\min_\theta\; \mathbb{E}_{(x,y)}\left[
\max_{\delta\in S(x)}\mathcal{L}(f_\theta(x+\delta),y)
\right],
\qquad
S(x)=\{\delta:\|\delta\|_\infty\le\epsilon,\;x+\delta\in[0,1]^d\}.
\]

**İçteki max:** model ağırlıkları sabitken görüntüyü değiştirip kaybı büyüt. **Dıştaki min:** bu zor görüntüyü kullanıp ağırlıkları değiştirerek kaybı azalt. Veri beklentisini minibatch'lerle, içteki aramayı PGD ile yaklaşık çözüyoruz. Madry ve arkadaşlarının temel yaklaşımı dayanıklı optimizasyon çerçevesidir. [Makale](https://arxiv.org/abs/1706.06083)

L∞ uzaklığı, pikseller arasındaki en büyük mutlak farktır: `max(abs(x_adv - x))`. Bizde pikseller `[0,1]`, ε ise `0.3`. Bu sınır toplam değişimi değil **her pikselin** değişimini sınırlar. Çözünürlük, veri ölçeği veya tehdit normu değişirse aynı epsilon başka anlama gelebilir.

## 2. FGSM neden gradyanın işaretini kullanıyor?

Bir görüntünün yakınında kaybı birinci dereceden Taylor yaklaşımıyla yazalım:

\[
\mathcal{L}(x+\delta)\approx\mathcal{L}(x)+g^\top\delta,
\qquad g=\nabla_x\mathcal{L}(f_\theta(x),y).
\]

L∞ topunda bu doğrusal ifadenin maksimumu:

\[
\max_{\|\delta\|_\infty\le\epsilon}g^\top\delta
=\epsilon\sum_i |g_i|=\epsilon\|g\|_1,
\qquad \delta_i^*=\epsilon\,\operatorname{sign}(g_i).
\]

Her pikseli gradyan pozitifse artırmak, negatifse azaltmak doğrusal yaklaşımın kaybını büyütür. L∞ sınırının L1 gradyan normuyla bu ilişkisi **dual norm** ilişkisidir. Görüntü aralığını da koruyarak:

\[
x_{\rm FGSM}=\operatorname{clip}_{[0,1]}
\left(x+\epsilon\operatorname{sign}(g)\right).
\]

Taylor yaklaşımı yaklaşık olduğu için bu tek adım gerçek sinir ağında en kötü görüntüyü garanti etmez. FGSM'nin birincil kaynağı [Goodfellow, Shlens ve Szegedy](https://arxiv.org/abs/1412.6572).

## 3. PGD bu fikri tekrar ediyor

Rastgele bir başlangıç oluştur:

\[
x^{(0)}=\operatorname{clip}_{[0,1]}(x+u),\qquad
u_i\sim\mathcal{U}(-\epsilon,\epsilon).
\]

Sonra her adımda gradyanı yeniden hesapla:

\[
x^{(t+1)}=\Pi_{[x-\epsilon,x+\epsilon]\cap[0,1]^d}
\left(x^{(t)}+\alpha\operatorname{sign}
\big(\nabla_{x^{(t)}}\mathcal{L}(f_\theta(x^{(t)}),y)\big)\right).
\]

`Π` burada piksel bazında kırpmadır. Merkez daima **orijinal x** kalır; merkezi her adım kaydırırsak ε sınırını aşabiliriz. Adım boyu `α`, toplam izin verilen değişim `ε` ile aynı şey değildir. Bu güncelleme ve rastgele başlangıç [yazarların PGD kodunda](https://github.com/MadryLab/mnist_challenge/blob/master/pgd_attack.py) doğrudan görülebilir.

Bir **restart**, yeni bir rastgele başlangıçla saldırıyı baştan çalıştırır. Farklı başlangıçlar farklı zor bölgeler bulabilir. Projemizde değerlendirme bütün başlangıç/ara adım/restart adaylarında bulunan yanlış sınıflandırmayı korur. Son adımın tesadüfen tekrar doğru olması önceki saldırı başarısını silmez. Bu, resmî kodun son adayı döndürme davranışına göre ek bir değerlendirme önlemidir.

## 4. Neden saldırıda `detach`, sonra ayrı eğitim gradyanı?

İki farklı türev alıyoruz:

1. Saldırıda `∇x loss`: model sabit, değişken görüntü. Yeni saldırı adımı `detach()` edilir ve bir sonraki adım için yeniden `requires_grad` açılır. Böylece bütün PGD iterasyonlarını tek dev hesaplama grafiğine bağlamayız.
2. Eğitimde `∇θ loss`: üretilen saldırılı görüntü sabit girdi kabul edilir; yeni ileri geçişte model ağırlıklarına türev alınır ve optimizer günceller.

Bu uygulamada dış adım, PGD algoritmasının bütün hesaplamalarından geriye türev almaz. `detach`, dış kaybın model ağırlıklarına türevini engellemez; yalnız saldırı üretim zincirinden ayırır. Resmî eğitim de önce adversarial örneği üretir, sonra bu örneği eğitim adımına besler. [Resmî eğitim kodu](https://github.com/MadryLab/mnist_challenge/blob/master/train.py)

İç maksimum tam çözülebilseydi, uygun düzenlilik koşullarında bu tür dış türev için Danskin yaklaşımı kullanılabilir. Burada PGD yaklaşık çözücüdür; bu teorik bağlantı kodun küresel optimum bulduğuna dair garanti değildir.

## 5. Doğrulukta payda bütün test seti

`N` test örneği için temiz doğruluk, doğru tahminlerin sayısının `N`'ye bölümüdür. PGD doğruluğunda da payda **aynı N** kalır:

\[
\widehat{A}_{\rm PGD}=\frac1N\sum_{i=1}^N
\mathbf{1}\{f(x_i)=y_i\}
\mathbf{1}\{\text{tutulan bütün PGD adayları doğru sınıflandırıldı}\}.
\]

Yalnız temizken doğru örnekleri paydaya alıp bunu genel dayanıklı doğruluk diye sunmuyoruz. Sıfır değişim de tehdit kümesinde olduğundan temizken yanlış bir örnek dayanıklı olamaz. `combined_accuracy`, temiz + FGSM + PGD adaylarının hepsini birlikte geçme oranıdır. Restart doğruluklarının ortalaması değildir.

Bu metrik **ampirik saldırı doğruluğudur**. PGD'nin hata bulması bir zayıflık gösterir; hata bulamaması izinli bütün değişimlerin güvenli olduğunu kanıtlamaz. Daha fazla adım/restart bu aramayı güçlendirir, sertifikaya dönüştürmez. Sonuçları yorumlarken `ε`, `α`, adım, restart, test örnek sayısı ve seed'i birlikte belirt.

Koddaki karşılıklar: `mnist_robust/attacks.py`, `mnist_robust/experiment.py`. İlk smoke çalışmasından sonra bu dosyaları yukarıdaki iki gradyan ayrımını takip ederek okumak iyi bir ikinci adımdır.


In [ ]:
%%writefile docs/references.bib
@inproceedings{madry2018towards,
  title     = {Towards Deep Learning Models Resistant to Adversarial Attacks},
  author    = {Madry, Aleksander and Makelov, Aleksandar and Schmidt, Ludwig and Tsipras, Dimitris and Vladu, Adrian},
  booktitle = {International Conference on Learning Representations},
  year      = {2018},
  eprint    = {1706.06083},
  archivePrefix = {arXiv},
  primaryClass = {stat.ML},
  url       = {https://arxiv.org/abs/1706.06083}
}

@misc{madrylab_mnist_challenge,
  author       = {{MadryLab}},
  title        = {{MNIST Adversarial Examples Challenge}: Official Implementation},
  howpublished = {\url{https://github.com/MadryLab/mnist_challenge}},
  url          = {https://github.com/MadryLab/mnist_challenge},
  note         = {Accessed 2026-10-24; use a pinned commit when reporting the final experiment}
}

@article{goodfellow2014explaining,
  title   = {Explaining and Harnessing Adversarial Examples},
  author  = {Goodfellow, Ian J. and Shlens, Jonathon and Szegedy, Christian},
  journal = {arXiv preprint arXiv:1412.6572},
  year    = {2014},
  url     = {https://arxiv.org/abs/1412.6572}
}


In [ ]:
%%writefile docs/VALIDATION.md
# Yerel doğrulama kaydı — 24 Ekim 2026

Bu kayıt yazılım kontrolüdür; makalenin ana sonucunun tekrarlandığı anlamına gelmez.

- `python -m unittest discover -s tests -v`: **17 test geçti**.
- Gerçek MNIST ile `smoke`, seed 0: baseline ve PGD adversarial eğitim, temiz/FGSM/PGD değerlendirme, checkpoint, CSV ve görseller başarıyla üretildi (CPU).
- Bağımsız notebook'un bütün çalışma hücreleri yukarıdan aşağı, hata olmadan çalıştırıldı. İzole klasörde kaynaklar notebook tarafından oluşturuldu; daha önce indirilen gerçek MNIST önbelleği kullanıldı. Notebook `auto` cihaz seçimiyle Apple MPS üzerinde çalıştı. İlk sürüm 30 hücreydi; üç kaynak-belge hücresi eklenen son sürüm 33 hücredir.
- Epoch sınırında kesilip devam edilen CPU adversarial eğitim, kesintisiz eğitimle aynı ağırlıkları verdi.
- Saldırı sınırları, temiz girişteki hataların korunması, restart/ara-adım başarısızlıklarının kaybolmaması, model modunun ve gradientlerin korunması test edildi.
- Farklı protokollerin ayrı gruplanması, örnek standart sapması (`ddof=1`) ve yinelenen seed reddi doğrulandı.

Test ortamı: Python 3.13.0, PyTorch 2.14.1, torchvision 0.29.1, NumPy 2.4.4, pandas 3.0.2, matplotlib 3.10.9. İlk 13 çekirdek test ayrıca PyTorch 2.11.0 CPU ile geçti. Bu sürümleri Colab'a zorla kurmayın; platformun uyumlu CUDA paket çiftini koruyun. Her koşunun `environment.json` ve `sessions.jsonl` dosyalarını saklayın.

Henüz yapılmayanlar: Colab/Kaggle CUDA çalıştırması, pilot/main GPU eğitimleri, üç seed'in bilimsel karşılaştırması, final PGD100×20 testi. MPS'te smoke tamamlandı; MPS/CUDA için bit düzeyinde resume eşitliği ayrıca doğrulanmadı. Birden çok data worker yerelde test edilmedi; varsayılan `num_workers=0`.

Smoke yalnızca 256 eğitim / 64 doğrulama / 64 test örneği, bir epoch kullanır. Başarı oranları ana deney sonucu olarak rapora taşınmamalıdır. Tam deney sonuçları elde edilene kadar raporun bulgular bölümü açık kalır.


## 4 · Önce eğit ve doğrula

Her mode/seed ayrı klasöre kaydedilir. Bu hücre **test setini açmaz**. Baseline temiz doğrulama, PGD eğitimli model saldırılı doğrulama üzerinden en iyi checkpoint'i seçer. `last.pt` devam etmek, `best.pt` daha sonra test etmek içindir.

PGD eğitimi aynı epoch sayısında normal eğitimden çok daha pahalıdır. Pilot süresini ölçmeden ana deneyin süresini varsayma. Önbelleğe alınmamış MNIST ilk seferde indirilecektir; sentetik smoke kullanılırsa bunun gerçek veri sonucu olmadığını çıktıda koruyoruz.


In [ ]:
import time

def run_command(args):
    print("\nÇalıştırılıyor:", " ".join(str(part) for part in args), flush=True)
    subprocess.run([str(part) for part in args], cwd=PROJECT_DIR, check=True)

training_wall_seconds = {}
for mode in MODES:
    for seed in SEEDS:
        run_dir = RUN_ROOT / RUN_CONFIG_NAME / mode / f"seed_{seed}"
        args = [sys.executable, "-m", "mnist_robust.experiment",
                "--config", f"configs/{CONFIG_NAME}.json", "--mode", mode,
                "--seed", str(seed), "--output-dir", str(RUN_ROOT),
                "--device", "auto", "--train-only",
                "--epochs-per-call", str(EPOCHS_PER_CALL)]
        if RESUME and (run_dir / "last.pt").exists():
            args.append("--resume")
        if SYNTHETIC:
            args.append("--synthetic")
        started = time.perf_counter()
        run_command(args)
        training_wall_seconds[f"{mode}/seed_{seed}"] = time.perf_counter() - started
import csv
config_epochs = __import__("json").loads(Path(f"configs/{CONFIG_NAME}.json").read_text())["epochs"]
TRAINING_COMPLETE = True
for mode in MODES:
    for seed in SEEDS:
        history_path = RUN_ROOT / RUN_CONFIG_NAME / mode / f"seed_{seed}" / "history.csv"
        completed_epochs = len(list(csv.DictReader(history_path.open()))) if history_path.exists() else 0
        TRAINING_COMPLETE &= completed_epochs >= config_epochs
        print(f"{mode}/seed_{seed}: {completed_epochs}/{config_epochs} epoch")
print("Bütün eğitimler tamam mı?", TRAINING_COMPLETE)
print("Bu notebook oturumundaki eğitim/yeniden başlatma süreleri (s):", training_wall_seconds)
print("Ekran görüntüsü: eğitim ve doğrulama çıktıları. Checkpoint'ler:", RUN_ROOT)


## 5 · Temiz, FGSM ve güçlü PGD testi

Smoke'ta küçük test otomatik açık. Pilot/main'de `PROTOCOL_FROZEN=True` gereklidir; önceki hücrelerin tamamı çalışabilir ama bu izin verilmeden test yapılmaz.

Bir örnek herhangi bir restart/PGD adımında yanlış sınıflandırıldıysa saldırı başarılı sayılır. Restart doğruluklarının ortalamasını almak fazla iyimser sonuç verir. Temizken yanlış olan bir örnek de dayanıklı sayılmaz. FGSM tek adımdır; PGD yinelemelidir. Final PGD100×20, `main` eğitim PGD40'tan güçlüdür ve pahalı olabilir.

Bu test seçilen saldırı altında ampirik dayanıklılık ölçer; bütün olası saldırılara karşı garanti vermez.


In [ ]:
EVALUATION_COMPLETED = False
if not TRAINING_COMPLETE:
    print("Eğitim parçası tamamlandı. RESUME=True ile eğitim hücresini tekrar çalıştırın; test bekliyor.")
elif not RUN_TEST_EVALUATION:
    print("Test kapalı: doğrulama sonuçlarıyla karar verin. Sonra PROTOCOL_FROZEN=True yapın.")
else:
    for mode in MODES:
        for seed in SEEDS:
            checkpoint = RUN_ROOT / RUN_CONFIG_NAME / mode / f"seed_{seed}" / "best.pt"
            if not checkpoint.exists():
                raise FileNotFoundError(f"Önce eğitim hücresini tamamlayın: {checkpoint}")
            args = [sys.executable, "-m", "mnist_robust.experiment",
                    "--config", f"configs/{CONFIG_NAME}.json", "--mode", mode,
                    "--seed", str(seed), "--output-dir", str(RUN_ROOT), "--device", "auto",
                    "--eval-only", "--checkpoint", str(checkpoint)]
            if SYNTHETIC:
                args.append("--synthetic")
            if FINAL_STRONG_EVAL:
                args.extend(["--eval-steps", "100", "--eval-restarts", "20"])
            run_command(args)
    EVALUATION_COMPLETED = True
    print("Test tamamlandı. Tabloyu görmeden önce config/seed/PGD bütçesini tekrar kontrol edelim.")


## 6 · CSV, tablolar ve şekiller

Her seed'in değerleri ayrıca saklanır. Ana deneyde **ortalama ± örnek standart sapma** kullanılır. Tek seed'in standart sapması hesaplanabilir bir belirsizlik ölçüsü değildir. Smoke/pilot/main veya farklı saldırı protokolleri aynı ortalamaya girmez.

**Screenshot hatırlatması:** koşul adının göründüğü sonuç tablosu, doğruluk karşılaştırması ve saldırılmış örnekler. CSV'leri de sakla; görüntü dosyası veri dosyasının yerine geçmez. Henüz koşulmamış deneyin sayısını rapora yazmayacağız.


In [ ]:
from IPython.display import display, Image, Markdown

if EVALUATION_COMPLETED:
    args = [sys.executable, "-m", "mnist_robust.summarize",
            "--runs", str(RUN_ROOT / RUN_CONFIG_NAME), "--output", str(RESULTS_DIR)]
    if len(SEEDS) < 2:
        args.append("--allow-single-seed")
    run_command(args)
    report = RESULTS_DIR / "summary.md"
    if report.exists():
        display(Markdown(report.read_text(encoding="utf-8")))
    for filename in ["accuracy.png", "training_curves.png"]:
        figure = RESULTS_DIR / filename
        if figure.exists():
            display(Image(filename=str(figure)))
    for mode in MODES:
        example = RUN_ROOT / RUN_CONFIG_NAME / mode / f"seed_{SEEDS[0]}" / "examples.png"
        if example.exists():
            display(Markdown(f"**{CONFIG_NAME} · {mode} · seed {SEEDS[0]}**"))
            display(Image(filename=str(example)))
    print("Sonuç dosyaları:", [p.name for p in sorted(RESULTS_DIR.glob("*"))])
else:
    print("Test tablosu yok. Eğitim geçmişi ve checkpoint'ler saklandı:", RUN_ROOT)
    for history in sorted((RUN_ROOT / RUN_CONFIG_NAME).glob("*/seed_*/history.csv")):
        print("\n", history.relative_to(RUN_ROOT))
        print(history.read_text(encoding="utf-8")[-3000:])


## 7 · Çalışmayı kaybetmeden sakla

Aşağıdaki arşiv kaynak kodunu, config'leri, checkpoint'leri ve sonuçları içerir. Uzun deneylerde arşiv büyüyebilir; Drive kaydı varsa orijinal klasör de kalıcıdır. Kaggle'da **Save Version** yapıp Output bölümünü sakla. Colab'da aşağıdaki isteğe bağlı indirme hücresini çalıştır.

Notebook'un kendisini de `.ipynb` olarak indirip public GitHub deposuna ekle. Veri ve büyük checkpoint arşivlerini Git'e yükleme; README'de ayrı bir paylaşım bağlantısı kullan.


In [ ]:
import json
import shutil
import zipfile
from datetime import datetime, timezone

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "config_name": CONFIG_NAME, "modes": MODES, "seeds": SEEDS,
    "protocol_frozen": PROTOCOL_FROZEN, "final_strong_eval": FINAL_STRONG_EVAL,
    "synthetic": SYNTHETIC,
    "note": "Smoke is a pipeline check, not a scientific result.",
    "session_training_seconds": training_wall_seconds,
}
(ARTIFACT_ROOT / "notebook_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)
archive_path = BASE_DIR / f"madry_mnist_{CONFIG_NAME}_artifacts.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for root, prefix in [(PROJECT_DIR, "project"), (ARTIFACT_ROOT, "artifacts")]:
        for p in sorted(root.rglob("*")):
            if not p.is_file() or "__pycache__" in p.parts or p.suffix == ".pyc":
                continue
            if "data" in p.relative_to(root).parts:  # MNIST yeniden indirilebilir
                continue
            if p.resolve() == archive_path.resolve():
                continue
            archive.write(p, f"{prefix}/{p.relative_to(root).as_posix()}")
print("Arşiv hazır:", archive_path, f"({archive_path.stat().st_size / 1024**2:.1f} MB)")
if IS_COLAB:
    print("İndirmek için sonraki hücrede DOWNLOAD_ARCHIVE=True yapın.")
elif IS_KAGGLE:
    print("Kaggle: Save Version → Output dosyalarını saklayın.")
else:
    from IPython.display import FileLink
    display(FileLink(str(archive_path)))


In [ ]:
DOWNLOAD_ARCHIVE = False
if DOWNLOAD_ARCHIVE and IS_COLAB:
    from google.colab import files
    files.download(str(archive_path))


## Sonraki adımı veriye göre seçelim

- **Smoke tamamlandıysa:** cihaz/config/tablo ekran görüntüsünü ve arşivi sakla; pilot normal eğitimle başla.
- **Pilot tamamlandıysa:** öğrenme eğrilerini ve süreyi karşılaştır; main bütçesini kararlaştır, seed listesini sabitle.
- **Üç seed'li main eğitimleri tamamlandıysa:** ayarları dondur, bütün checkpoint'leri aynı PGD100×20 ile test et.
- **Gerçek sonuçlar hazırsa:** seed tablosu, ortalama ± std, eğitim maliyeti ve makaleden farkları raporla. Gerçek ders LaTeX şablonunu kullan; public repo bağlantısını ekle.

Sunumda her grup üyesi şu zinciri açıklayabilsin: **tehdit modeli → min–max → PGD → adversarial training → güçlü değerlendirme → sonuç ve sınırlılıklar**.
